# Modelado de Propagación de la Tuberculosis

**Proyecto 1 — Ecuaciones Diferenciales: Modelación**
Universidad Anáhuac México · Facultad de Ingeniería

Este cuaderno ejecuta la simulación completa del proyecto. Corre las celdas en orden.

---

### Los dos modelos

**Modelo A — transmisión (logístico, no lineal)**

$$\frac{dI}{dt} = \beta I\left(1 - \frac{I}{N}\right) - \gamma I = r\,I\left(1 - \frac{I}{K}\right)$$

**Modelo B — balance (lineal)**

$$\frac{dI}{dt} = \lambda - \delta I$$

## 1. Preparación

En Google Colab, la siguiente celda clona el repositorio e instala las dependencias.
Si ya estás ejecutando desde una copia local, simplemente la salta.

In [ ]:
import os, sys, subprocess

REPO = "tb-modelado-ed"
URL  = "https://github.com/USUARIO/tb-modelado-ed.git"   # <-- cambiar por la URL real

if not os.path.exists("src"):
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", URL], check=True)
    os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.path.abspath("src"))
print("Listo. Directorio de trabajo:", os.getcwd())

## 2. La base de datos

In [ ]:
from estimacion import cargar_datos
import numpy as np

anio, t, inc, casos = cargar_datos()

print(f"Periodo: {int(anio[0])}-{int(anio[-1])}  ({len(t)} observaciones)")
print(f"Incidencia: min {inc.min():.0f}, max {inc.max():.0f}, media {inc.mean():.2f}")
print()
for a, i in zip(anio[:5], inc[:5]):
    print(f"  {int(a)}: {i:.0f} casos por 100 000")
print("  ...")

> **Nota metodológica.** La incidencia es un *flujo* (casos nuevos por año); la variable $I(t)$ de los modelos
> es un *acervo* (prevalencia). En equilibrio se relacionan por $I^* = \lambda/\delta = \lambda D$, donde $D$ es
> la duración promedio del caso.

## 3. Estimación de parámetros

Primero por regresión lineal sobre la forma linealizada de cada modelo, que es el método reproducible en Excel y en la calculadora.

In [ ]:
from estimacion import estimar_B_lineal, estimar_A_lineal

rB = estimar_B_lineal(t, inc)
rA = estimar_A_lineal(t, inc)

print(f"Modelo B:  delta = {rB['delta']:.4f} /año   lambda = {rB['lambda']:.3f}   I* = {rB['I_eq']:.2f}   R2 = {rB['R2']:.3f}")
print(f"Modelo A:  r     = {rA['r']:.4f} /año   K      = {rA['K']:.2f}                  R2 = {rA['R2']:.3f}")
print()
print("R2 bajos porque la derivada por diferencias finitas es ruidosa,")
print(f"pero ambos coinciden en el equilibrio: {rB['I_eq']:.1f} vs {rA['K']:.1f}.")

Ahora el ajuste no lineal de la solución analítica directamente a los datos, que es más preciso porque no deriva nada.

In [ ]:
from estimacion import ajustar_B, ajustar_A

fB = ajustar_B(t, inc)
fA = ajustar_A(t, inc)

print("MODELO B")
print(f"  lambda = {fB['lambda']:.3f} +/- {fB['err_lambda']:.3f}")
print(f"  delta  = {fB['delta']:.4f} +/- {fB['err_delta']:.4f} /año")
print(f"  I*     = {fB['I_eq']:.2f} por 100 000")
print(f"  tau    = {fB['modelo'].tau:.2f} años")
print(f"  R2 = {fB['R2']:.4f}   RMSE = {fB['rmse']:.2f}")
print()
print("MODELO A")
print(f"  r = {fA['r']:+.4f} /año   K = {fA['K']:.3e}  <- sin sentido físico")
print(f"  R2 = {fA['R2']:.4f}   RMSE = {fA['rmse']:.2f}")
print()
print(f"Referencia (media constante): RMSE = {np.sqrt(((inc-inc.mean())**2).mean()):.2f}")

### Conclusión del ajuste

Los datos **seleccionan al Modelo B**. La fase de crecimiento logístico que describe el Modelo A no es
observable en la serie nacional: México no atravesó una invasión epidémica durante el periodo, sino una
relajación hacia un equilibrio más bajo.

## 4. Simulación de un brote en entorno cerrado

Aquí sí aplica el Modelo A: un centro penitenciario cumple las hipótesis de población cerrada y mezcla homogénea.

In [ ]:
from modelos import ModeloA

N, gamma, I0 = 1000.0, 1/3, 1.0

for R0 in [3.0, 2.0, 1.5, 0.8]:
    A = ModeloA(beta=R0*gamma, gamma=gamma, N=N)
    if R0 > 1:
        print(f"R0 = {R0:4.1f}  ->  r = {A.r:+.3f}   K = {A.K:6.1f} casos ({A.K/N*100:.1f} % de la población)")
    else:
        print(f"R0 = {R0:4.1f}  ->  r = {A.r:+.3f}   el brote se extingue")

A3 = ModeloA(beta=3*gamma, gamma=gamma, N=N)
print()
print("Trayectoria con R0 = 3:")
for y in [1, 5, 10, 15, 20]:
    v = float(A3.solucion(y, I0))
    print(f"  t = {y:2d} años  ->  I = {v:6.1f} casos ({v/N*100:.1f} %)")

## 5. Validación numérica

Se comparan Euler y Runge-Kutta de cuarto orden contra la solución analítica.

In [ ]:
from modelos import tabla_error

for nombre, modelo, y0 in [("Modelo A", A3, I0), ("Modelo B", fB["modelo"], fB["I0"])]:
    exacta, filas = tabla_error(modelo, y0, 10.0)
    print(f"{nombre} - exacta en t=10: {exacta:.6f}")
    print(f"  {'h':>9} {'error Euler':>13} {'razón':>7} {'error RK4':>13} {'razón':>7}")
    for f in filas:
        rE = f"{f['razon_euler']:.2f}" if f['razon_euler'] else "  -"
        rR = f"{f['razon_rk4']:.2f}"   if f['razon_rk4']   else "  -"
        print(f"  {f['h']:9.5f} {f['error_euler']:13.3e} {rE:>7} {f['error_rk4']:13.3e} {rR:>7}")
    print()

print("Euler reduce el error x2 (orden 1); RK4 lo reduce x16 = 2^4 (orden 4).")

## 6. Figuras

Genera las siete figuras del documento y las muestra.

In [ ]:
import figuras
from IPython.display import Image, display

for nombre in figuras.generar_todas():
    print(nombre)
    display(Image(filename="figuras/" + nombre, width=760))

## 7. Reporte completo

La celda siguiente ejecuta todo el análisis de una sola vez, igual que el flujo de integración continua.

In [ ]:
!python src/main.py